In [ ]:

#Iteration 7
# Heterogeneous Ensemble
# Old strategy: ResNet18 + ResNet18 (same arch, same 128px, same weak aug) -> low diversity, +0.5-1% max
# New strategy: ResNet34 + EfficientNet-B0 (different archs) + 224px + Strong Aug + Mixup + Label Smoothing -> +3-5% expected

# Key wins:
# 1. Diversity > Seed: Two different architectures make different mistakes, averaging helps more
# 2. 224px vs 128px: Keeps facial details (eyes, mouth) for emotion
# 3. Strong Aug: RandAugment + RandomErasing simulates occlusion/lighting
# 4. Mixup + Label Smoothing: Fixes overconfidence on imbalanced FER classes (disgust/fear are rare)
# 5. AdamW + Cosine + AMP: Better regularization + faster convergence

import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
import timm # EfficientNet via timm is lighter than torchvision; fallback to torchvision if missing

# --------------------------------------------
# 0. Config - CHANGES vs safe_ensemble
# --------------------------------------------
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR_CANDIDATES = ["/kaggle/input/competitions/iitb-itc-aiml-team-selection", "/kaggle/input"]
DATA_DIR = next((d for d in DATA_DIR_CANDIDATES if os.path.exists(os.path.join(d, "train.csv"))), None)
if DATA_DIR is None:
    for root, _, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root
            break
if DATA_DIR is None: raise FileNotFoundError("train.csv not found")
print(f"Dataset found at: {DATA_DIR}")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# UPGRADE 1: 224px instead of 128px in safe_ensemble:65
IMG_SIZE = 224
BATCH_SIZE = 32  # lower because 224px uses more VRAM, but more stable
EPOCHS = 25
LR = 3e-4  # higher for AdamW + EfficientNet

def image_path(row_id, img_dir):
    return os.path.join(img_dir, row_id if str(row_id).endswith(".jpg") else f"{row_id}.jpg")

# --------------------------------------------
# 1. Data - Same split but STRONGER aug
# --------------------------------------------
full_train_df = pd.read_csv(TRAIN_CSV)
train_df, val_df = train_test_split(full_train_df, test_size=0.15, stratify=full_train_df["label"], random_state=42)
print(f"Train: {len(train_df)} Val: {len(val_df)}")
print(full_train_df["label"].value_counts()) # check imbalance

class FERDataset(Dataset):
    def __init__(self, df, img_dir, transform, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.has_labels = has_labels
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Keep L->RGB convert but now at 224px we preserve detail
        img = Image.open(image_path(row["id"], self.img_dir)).convert("L").convert("RGB")
        img = self.transform(img)
        if self.has_labels: return img, CLASS_TO_IDX[row["label"]]
        return img, row["id"]

# UPGRADE 2: Strong augmentation vs safe_ensemble:73-78 (only flip/rotate 10deg)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandAugment(num_ops=2, magnitude=9), # NEW: auto strong aug
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet stats, not 0.5
    transforms.RandomErasing(p=0.25), # NEW: simulates occlusion (hand over mouth)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
tta_flip_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = FERDataset(train_df, TRAIN_IMG_DIR, train_transform)
val_ds = FERDataset(val_df, TRAIN_IMG_DIR, eval_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Handle imbalance better than simple weight average
class_counts = train_df["label"].value_counts().reindex(CLASSES).values
weights = 1.0 / class_counts
weights = weights / weights.sum() * len(CLASSES)
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

# UPGRADE 3: Mixup function (mixes two images/labels) - huge for FER generalization
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = torch.distributions.Beta(alpha, alpha).sample().item()
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# --------------------------------------------
# 2. Models - HETEROGENEOUS (beats same-arch ensemble)
# --------------------------------------------
def get_resnet34():
    m = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, len(CLASSES)))
    return m

def get_efficientnet_b0():
    try:
        m = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(CLASSES), drop_rate=0.3)
    except:
        # fallback to torchvision
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
    return m

model_a = get_resnet34().to(DEVICE)        # Strong CNN with residual depth
model_b = get_efficientnet_b0().to(DEVICE) # Different family: MBConv + SE, better at fine textures

# UPGRADE 4: Label Smoothing + AdamW vs safe_ensemble:98 (plain CE + Adam)
criterion_a = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
criterion_b = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer_a = torch.optim.AdamW(model_a.parameters(), lr=LR, weight_decay=1e-4)
optimizer_b = torch.optim.AdamW(model_b.parameters(), lr=LR, weight_decay=1e-4)
scheduler_a = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_a, T_max=EPOCHS)
scheduler_b = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

def train_one_model(model, optimizer, scheduler, criterion, name, save_path):
    best_acc = 0
    for epoch in range(EPOCHS):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            # Apply Mixup 50% of batches
            use_mixup = torch.rand(1).item() > 0.5
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=scaler is not None):
                if use_mixup:
                    mixed_imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=0.2)
                    out = model(mixed_imgs)
                    loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
                else:
                    out = model(imgs)
                    loss = criterion(out, labels)
            if scaler:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        scheduler.step()
        val_acc = evaluate(model, val_loader)
        print(f"[{name}] Epoch {epoch+1}/{EPOCHS} - val_acc: {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  -> New best {name}: {best_acc:.4f} saved.")
    return best_acc

print("Training Model A: ResNet34...")
best_a = train_one_model(model_a, optimizer_a, scheduler_a, criterion_a, "ResNet34", "best_resnet34.pth")
print("Training Model B: EfficientNet-B0...")
best_b = train_one_model(model_b, optimizer_b, scheduler_b, criterion_b, "EfficientNet-B0", "best_effnet_b0.pth")
print(f"Best Val: ResNet34={best_a:.4f}, EffNet={best_b:.4f}")

# --------------------------------------------
# 3. Ensemble Predict - Weighted Soft Voting + TTA
# --------------------------------------------
model_a.load_state_dict(torch.load("best_resnet34.pth", map_location=DEVICE))
model_b.load_state_dict(torch.load("best_effnet_b0.pth", map_location=DEVICE))
model_a.eval(); model_b.eval()

# Weight by validation accuracy (better model gets more vote)
total_best = best_a + best_b
w_a, w_b = best_a / total_best, best_b / total_best
print(f"Ensemble weights: ResNet34={w_a:.2f}, EffNet={w_b:.2f}")

test_df = pd.read_csv(TEST_CSV)
test_ds_normal = FERDataset(test_df, TEST_IMG_DIR, eval_transform, has_labels=False)
test_ds_flip = FERDataset(test_df, TEST_IMG_DIR, tta_flip_transform, has_labels=False)
test_loader_normal = DataLoader(test_ds_normal, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_flip = DataLoader(test_ds_flip, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_ids, all_preds = [], []
with torch.no_grad():
    for (imgs_n, ids), (imgs_f, _) in zip(test_loader_normal, test_loader_flip):
        imgs_n, imgs_f = imgs_n.to(DEVICE), imgs_f.to(DEVICE)
        # Average 4 forward passes: A_normal + A_flip + B_normal + B_flip, weighted
        probs_a_n = torch.softmax(model_a(imgs_n), dim=1)
        probs_a_f = torch.softmax(model_a(imgs_f), dim=1)
        probs_b_n = torch.softmax(model_b(imgs_n), dim=1)
        probs_b_f = torch.softmax(model_b(imgs_f), dim=1)
        
        probs_a = (probs_a_n + probs_a_f) / 2
        probs_b = (probs_b_n + probs_b_f) / 2
        probs_final = w_a * probs_a + w_b * probs_b  # weighted average beats simple /4 in safe_ensemble:148
        
        preds = probs_final.argmax(dim=1).cpu().numpy()
        all_ids.extend(ids)
        all_preds.extend([IDX_TO_CLASS[p] for p in preds])

submission = pd.DataFrame({"id": all_ids, "label": all_preds})
submission.to_csv("submission_hetero_ensemble.csv", index=False)
print("Saved submission_hetero_ensemble.csv")


Dataset found at: /kaggle/input/competitions/iitb-itc-aiml-team-selection
Train: 24402 Val: 4307
label
happy       7215
neutral     4965
sad         4830
fear        4097
angry       3995
surprise    3171
disgust      436
Name: count, dtype: int64
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 166MB/s] 


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Training Model A: ResNet34...
[ResNet34] Epoch 1/25 - val_acc: 0.5322
  -> New best ResNet34: 0.5322 saved.
[ResNet34] Epoch 2/25 - val_acc: 0.4414
[ResNet34] Epoch 3/25 - val_acc: 0.5542
  -> New best ResNet34: 0.5542 saved.
[ResNet34] Epoch 4/25 - val_acc: 0.5765
  -> New best ResNet34: 0.5765 saved.
[ResNet34] Epoch 5/25 - val_acc: 0.5886
  -> New best ResNet34: 0.5886 saved.
[ResNet34] Epoch 6/25 - val_acc: 0.6023
  -> New best ResNet34: 0.6023 saved.
[ResNet34] Epoch 7/25 - val_acc: 0.5830
[ResNet34] Epoch 8/25 - val_acc: 0.6450
  -> New best ResNet34: 0.6450 saved.
[ResNet34] Epoch 9/25 - val_acc: 0.6448
[ResNet34] Epoch 10/25 - val_acc: 0.6269
[ResNet34] Epoch 11/25 - val_acc: 0.6564
  -> New best ResNet34: 0.6564 saved.
[ResNet34] Epoch 12/25 - val_acc: 0.6550
[ResNet34] Epoch 13/25 - val_acc: 0.6596
  -> New best ResNet34: 0.6596 saved.
[ResNet34] Epoch 14/25 - val_acc: 0.6680
  -> New best ResNet34: 0.6680 saved.
[ResNet34] Epoch 15/25 - val_acc: 0.6678
[ResNet34] Epoch 16/25

In [ ]:
#Iteration-8

#   1. WeightedRandomSampler added to the training DataLoader — on top of
#      your existing class-weighted loss, this changes WHICH images get
#      shown each epoch, so disgust/fear are physically seen far more often.
#   2. train_acc / val_acc / gap logged per epoch, per model — same
#      diagnostic you already know how to read, and useful content for
#      your report.
#   3. Batch-level heartbeat prints every 100 batches — cheap insurance
#      against a silent hang costing you time again.
#   4. Optional THIRD model (ResNet18 — fast, already proven reliable in
#      your earlier runs) for extra ensemble diversity. Toggle with
#      USE_THIRD_MODEL below based on how much time you actually have.
#
# Everything that already worked is UNCHANGED: ResNet34 + EfficientNet-B0,
# 224px, RandAugment + RandomErasing + Mixup, AdamW + Cosine + AMP, label
# smoothing, weighted soft-vote TTA ensembling.


import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
import timm

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

USE_THIRD_MODEL = True  # set False to save real time — trains just ResNet34 + EffNet-B0

# --------------------------------------------
# 0. Paths
# --------------------------------------------
DATA_DIR_CANDIDATES = ["/kaggle/input/competitions/iitb-itc-aiml-team-selection", "/kaggle/input"]
DATA_DIR = next((d for d in DATA_DIR_CANDIDATES if os.path.exists(os.path.join(d, "train.csv"))), None)
if DATA_DIR is None:
    for root, _, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root
            break
if DATA_DIR is None:
    raise FileNotFoundError("train.csv not found")
print(f"Dataset found at: {DATA_DIR}")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
LR = 3e-4

def image_path(row_id, img_dir):
    return os.path.join(img_dir, row_id if str(row_id).endswith(".jpg") else f"{row_id}.jpg")

# --------------------------------------------
# 1. Data
# --------------------------------------------
full_train_df = pd.read_csv(TRAIN_CSV)
train_df, val_df = train_test_split(full_train_df, test_size=0.15, stratify=full_train_df["label"], random_state=42)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")

class FERDataset(Dataset):
    def __init__(self, df, img_dir, transform, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.has_labels = has_labels
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(image_path(row["id"], self.img_dir)).convert("L").convert("RGB")
        img = self.transform(img)
        if self.has_labels:
            return img, CLASS_TO_IDX[row["label"]]
        return img, row["id"]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
tta_flip_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = FERDataset(train_df, TRAIN_IMG_DIR, train_transform)
val_ds = FERDataset(val_df, TRAIN_IMG_DIR, eval_transform)

# --------------------------------------------
# NEW: WeightedRandomSampler — physically shows rare classes more often,
# in addition to the loss weighting below (two independent defenses
# against the disgust/fear imbalance instead of one).
# --------------------------------------------
class_counts = train_df["label"].value_counts().reindex(CLASSES).values
class_weights_for_loss = 1.0 / class_counts
class_weights_for_loss = class_weights_for_loss / class_weights_for_loss.sum() * len(CLASSES)
class_weights = torch.tensor(class_weights_for_loss, dtype=torch.float32).to(DEVICE)

per_class_sample_weight = dict(zip(CLASSES, 1.0 / class_counts))
sample_weights = train_df["label"].map(per_class_sample_weight).values
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = torch.distributions.Beta(alpha, alpha).sample().item()
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# --------------------------------------------
# 2. Models
# --------------------------------------------
def get_resnet34():
    m = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, len(CLASSES)))
    return m

def get_efficientnet_b0():
    try:
        m = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(CLASSES), drop_rate=0.3)
    except Exception:
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
    return m

def get_resnet18():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, len(CLASSES)))
    return m

model_configs = [
    {"name": "ResNet34", "model": get_resnet34().to(DEVICE), "path": "best_resnet34.pth"},
    {"name": "EfficientNet-B0", "model": get_efficientnet_b0().to(DEVICE), "path": "best_effnet_b0.pth"},
]
if USE_THIRD_MODEL:
    model_configs.append({"name": "ResNet18", "model": get_resnet18().to(DEVICE), "path": "best_resnet18_v3.pth"})

for cfg in model_configs:
    cfg["criterion"] = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    cfg["optimizer"] = torch.optim.AdamW(cfg["model"].parameters(), lr=LR, weight_decay=1e-4)
    cfg["scheduler"] = torch.optim.lr_scheduler.CosineAnnealingLR(cfg["optimizer"], T_max=EPOCHS)

scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

def train_one_model(cfg):
    model, optimizer, scheduler, criterion, name, save_path = (
        cfg["model"], cfg["optimizer"], cfg["scheduler"], cfg["criterion"], cfg["name"], cfg["path"]
    )
    best_acc = 0
    for epoch in range(EPOCHS):
        model.train()
        running_loss, train_correct, train_total = 0.0, 0, 0
        for batch_idx, (imgs, labels) in enumerate(train_loader):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            use_mixup = torch.rand(1).item() > 0.5
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=scaler is not None):
                if use_mixup:
                    mixed_imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=0.2)
                    out = model(mixed_imgs)
                    loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
                else:
                    out = model(imgs)
                    loss = criterion(out, labels)
            if scaler:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            if not use_mixup:  # only clean-label batches counted toward train_acc (mixup labels are blended)
                train_correct += (out.argmax(dim=1) == labels).sum().item()
                train_total += labels.size(0)

            if batch_idx % 100 == 0:
                print(f"  [{name}] epoch {epoch + 1}, batch {batch_idx}/{len(train_loader)}, loss so far: {loss.item():.4f}")

        scheduler.step()
        val_acc = evaluate(model, val_loader)
        train_acc = train_correct / train_total if train_total > 0 else float("nan")
        gap = train_acc - val_acc if train_total > 0 else float("nan")
        print(f"[{name}] Epoch {epoch + 1}/{EPOCHS} - train_acc: {train_acc:.4f} - val_acc: {val_acc:.4f} - gap: {gap:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  -> New best {name}: {best_acc:.4f} saved.")
    return best_acc

best_scores = {}
for cfg in model_configs:
    print(f"\nTraining {cfg['name']}...")
    best_scores[cfg["name"]] = train_one_model(cfg)

print("\nBest validation scores:", best_scores)

# --------------------------------------------
# 3. Ensemble Predict — Weighted Soft Voting + TTA across ALL trained models
# --------------------------------------------
for cfg in model_configs:
    cfg["model"].load_state_dict(torch.load(cfg["path"], map_location=DEVICE))
    cfg["model"].eval()

total_best = sum(best_scores.values())
ensemble_weights = {name: score / total_best for name, score in best_scores.items()}
print("Ensemble weights:", ensemble_weights)

test_df = pd.read_csv(TEST_CSV)
test_ds_normal = FERDataset(test_df, TEST_IMG_DIR, eval_transform, has_labels=False)
test_ds_flip = FERDataset(test_df, TEST_IMG_DIR, tta_flip_transform, has_labels=False)
test_loader_normal = DataLoader(test_ds_normal, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_flip = DataLoader(test_ds_flip, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_ids, all_preds = [], []
with torch.no_grad():
    for (imgs_n, ids), (imgs_f, _) in zip(test_loader_normal, test_loader_flip):
        imgs_n, imgs_f = imgs_n.to(DEVICE), imgs_f.to(DEVICE)
        probs_final = torch.zeros(imgs_n.size(0), len(CLASSES), device=DEVICE)
        for cfg in model_configs:
            model = cfg["model"]
            w = ensemble_weights[cfg["name"]]
            probs_n = torch.softmax(model(imgs_n), dim=1)
            probs_f = torch.softmax(model(imgs_f), dim=1)
            probs_final += w * (probs_n + probs_f) / 2
        preds = probs_final.argmax(dim=1).cpu().numpy()
        all_ids.extend(ids)
        all_preds.extend([IDX_TO_CLASS[p] for p in preds])

submission = pd.DataFrame({"id": all_ids, "label": all_preds})
submission.to_csv("submission_final_ensemble.csv", index=False)
print("Saved submission_final_ensemble.csv")

Dataset found at: /kaggle/input/competitions/iitb-itc-aiml-team-selection
Using device: cuda
Train: 24402  Val: 4307
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 177MB/s] 


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s] 



Training ResNet34...
  [ResNet34] epoch 1, batch 0/763, loss so far: 1.6837
  [ResNet34] epoch 1, batch 100/763, loss so far: 1.2623
  [ResNet34] epoch 1, batch 200/763, loss so far: 1.4129
  [ResNet34] epoch 1, batch 300/763, loss so far: 0.5974
  [ResNet34] epoch 1, batch 400/763, loss so far: 1.3446
  [ResNet34] epoch 1, batch 500/763, loss so far: 1.1162
  [ResNet34] epoch 1, batch 600/763, loss so far: 1.0316
  [ResNet34] epoch 1, batch 700/763, loss so far: 1.0366
[ResNet34] Epoch 1/25 - train_acc: 0.3033 - val_acc: 0.3973 - gap: -0.0940
  -> New best ResNet34: 0.3973 saved.
  [ResNet34] epoch 2, batch 0/763, loss so far: 1.6328
  [ResNet34] epoch 2, batch 100/763, loss so far: 1.1973
  [ResNet34] epoch 2, batch 200/763, loss so far: 0.6373
  [ResNet34] epoch 2, batch 300/763, loss so far: 0.7601
  [ResNet34] epoch 2, batch 400/763, loss so far: 1.1897
  [ResNet34] epoch 2, batch 500/763, loss so far: 1.0672
  [ResNet34] epoch 2, batch 600/763, loss so far: 1.0653
  [ResNet34] e